# CSET485 — AI and Society · Assignment 1## Stress-testing a credit-approval model: German Credit Data**Name:** Anurag Kumar  **Enrolment number:** S24CSEU1224### Dataset and seed assignment (from the assignment brief)| Rule | Value from enrolment number ||---|---|| Last digit of roll number | **4** → **German Credit Data** || Last 3 digits as `random_state` | **224** || Train/test split | **70% / 30%** |Target encoding required by the brief: **approval = 1 (approve), 0 (reject)**.  In the UCI codebook the raw target is `1 = Good credit`, `2 = Bad credit`, so`approval = 1 if target == 1 else 0`.### What this notebook producesEvery number, table and figure cited in the accompanying PDF report is generated byrunning this notebook top to bottom. Nothing is hard-coded.### A note on the statistics libraryCoefficient standard errors, z-statistics, p-values and 95% confidence intervals arecomputed from a **self-contained unpenalised logistic MLE** (Newton–Raphson with astep-halving line search, standard errors from the inverse observed Fisherinformation). This is the same estimator `statsmodels.Logit` implements, but writing itout means the notebook has **no dependency beyond numpy / pandas / scipy / scikit-learn /matplotlib**, so it reproduces anywhere. `sklearn.LogisticRegression` is deliberately**not** used for inference because it applies L2 regularisation by default, which biasescoefficients and invalidates the standard errors.

In [ ]:
import sys, io, json, warningsimport numpy as npimport pandas as pdfrom scipy import statsimport matplotlibimport matplotlib.pyplot as pltfrom sklearn.model_selection import train_test_split, StratifiedKFoldfrom sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_scorewarnings.filterwarnings("ignore")pd.set_option("display.width", 200)pd.set_option("display.max_columns", 60)SEED = 224          # last 3 digits of enrolment number S24CSEU1224TEST_SIZE = 0.30    # 70/30 split required by the briefnp.random.seed(SEED)NAME       = "Anurag Kumar"ENROLMENT  = "S24CSEU1224"COURSE     = "CSET485 - AI and Society, Assignment 1"print("="*62)print(COURSE)print("Name      :", NAME)print("Enrolment :", ENROLMENT)print("Dataset   : German Credit Data (last digit 4)")print("Seed      : random_state =", SEED, "(last 3 digits of enrolment)")print("="*62)print("numpy", np.__version__, "| pandas", pd.__version__)

---## 1. Load the dataThe file is the original UCI `german.data`: 1000 rows, 20 attributes + 1 target,space-separated, no header. The loader fetches it from UCI and falls back to a localcopy if the machine is offline. An integrity check confirms the expected shape and theknown 700/300 class split before anything else runs.

In [ ]:
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"LOCAL   = "german.data"   # fallback copy kept next to this notebookCOLS = ["checking_status","duration","credit_history","purpose","credit_amount",        "savings_status","employment_since","installment_rate","personal_status_sex",        "other_debtors","residence_since","property","age","other_installment_plans",        "housing","existing_credits","job","dependents","telephone","foreign_worker",        "target"]NUMERIC = ["duration","credit_amount","installment_rate","residence_since","age",           "existing_credits","dependents"]CATEGORICAL = [c for c in COLS if c not in NUMERIC + ["target"]]def load_german():    try:        df = pd.read_csv(UCI_URL, sep=r"\s+", header=None, names=COLS, engine="python")        print("loaded from UCI")    except Exception as e:        print("UCI unreachable (%s); using local copy" % type(e).__name__)        df = pd.read_csv(LOCAL, sep=r"\s+", header=None, names=COLS, engine="python")    return dfraw = load_german()# ---- integrity checks ----assert raw.shape == (1000, 21), f"unexpected shape {raw.shape}"assert raw["target"].value_counts().to_dict() == {1: 700, 2: 300}, "unexpected class split"print("shape:", raw.shape)print("target counts (1=Good, 2=Bad):", raw["target"].value_counts().sort_index().to_dict())raw.head()

---## 2. Data preprocessingThe brief requires explicit documentation of **missing values, categorical encoding,feature scaling and outliers**. Each is handled below and the reasoning is recordedalongside the code.

### 2.1 Missing values`german.data` contains **no `NaN` values** — but that is misleading, and treating it as"clean" would be the first mistake. Three attributes encode *absence of information* asan explicit category:| Attribute | Sentinel level | Codebook meaning | Share of rows ||---|---|---|---|| `checking_status` | `A14` | no checking account | 39.4% || `savings_status`  | `A65` | unknown / no savings account | 18.3% || `property`        | `A124`| unknown / no property | 15.4% |**Decision: keep these as their own categories rather than imputing them.**"No checking account" is a real, observable fact about an applicant, not a missingmeasurement, so imputing it would destroy information. This decision matters later:Part 2.3 shows that `A14` carries the single largest positive coefficient in the wholemodel, so the model is partly reasoning from *absence of data*.

In [ ]:
print("explicit NaN values in the dataset:", int(raw.isna().sum().sum()))print("duplicate rows:", int(raw.duplicated().sum()))print()for col, lvl, desc in [("checking_status","A14","no checking account"),                       ("savings_status","A65","unknown / no savings account"),                       ("property","A124","unknown / no property")]:    n = int((raw[col] == lvl).sum())    print(f"{col:18s} == {lvl:5s} ({desc:30s}): {n:4d} rows ({n/10:.1f}%)")

### 2.2 Target encoding and the derived `sex` variableAttribute 9 (`personal_status_sex`) entangles marital status with sex in a single field.Sex can be recovered cleanly, but marital status cannot: level `A92` lumps*divorced / separated / married* women into one category, whereas the male levels keepthose states apart. Sex is therefore derived for the fairness analysis, and the rawattribute is kept in the model. **This asymmetry is itself a dataset defect** and isdiscussed in the report.

In [ ]:
def add_derived(df):    df = df.copy()    df["approval"] = (df["target"] == 1).astype(int)   # 1 = approve (good), 0 = reject (bad)    sex_map = {"A91":"male",    # male   : divorced/separated               "A92":"female",  # female : divorced/separated/married               "A93":"male",    # male   : single               "A94":"male",    # male   : married/widowed               "A95":"female"}  # female : single  (absent from this dataset)    df["sex"] = df["personal_status_sex"].map(sex_map)    df["is_female"] = (df["sex"] == "female").astype(int)    return dfdf_all = add_derived(raw)print("approval rate overall: %.4f" % df_all["approval"].mean())print(df_all["personal_status_sex"].value_counts().sort_index().to_dict())print("\napproval rate by sex:")print(df_all.groupby("sex")["approval"].agg(["mean","count"]).round(4))

### 2.3 OutliersOutliers are assessed with the 1.5×IQR rule. Three numeric attributes carry a long righttail: `duration` (7.0% flagged, max 72 months), `credit_amount` (7.2%, max 18,424 DM) and`age` (2.3%, max 75 years).**Decision: no outlier is removed or winsorised.** Every flagged value is substantivelyplausible — a 72-month loan and a 75-year-old applicant are ordinary banking cases, notrecording errors. Deleting them would discard exactly the high-risk, long-durationapplicants the model most needs to get right, and would bias the fairness analysisbecause older applicants are disproportionately represented in the tail. The`dependents` "outliers" (15.5%) are an artefact of a binary 1/2 variable and aremeaningless under the IQR rule.

In [ ]:
rows = []for c in NUMERIC:    q1, q3 = raw[c].quantile([.25, .75]); iqr = q3 - q1    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr    n = int(((raw[c] < lo) | (raw[c] > hi)).sum())    rows.append(dict(attribute=c, min=raw[c].min(), median=raw[c].median(), max=raw[c].max(),                     iqr_low=lo, iqr_high=hi, n_outliers=n, pct=n/10))print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.2f}"))

### 2.4 Categorical encoding, and why sparse levels must be collapsed firstCategoricals are one-hot encoded with `drop_first=True` to avoid the dummy-variable trap.Encoding all levels naively **breaks the model**. Attribute 4 (`purpose`) has 10 levels,and `A48` (retraining) has only 9 rows in the whole dataset — of which, under seed 224,just **2 land in the training split, and both are approved**. That is *perfectseparation*: the maximum-likelihood estimate for that coefficient diverges to +∞,Newton–Raphson fails to converge after 200 iterations, the largest fitted probabilityhits 1.0000, and every standard error in the model becomes meaningless. The cell belowreports the training-split counts that cause this.**Decision: collapse sparse levels into semantically coherent groups** (not an arbitrary"other" bucket) before encoding:| Attribute | Merged | Rationale ||---|---|---|| `credit_history` | `A30` + `A31` → `A30_31` | both mean "no prior repayment problems" || `purpose` | `A44`,`A45`,`A48`,`A410` → `A4_other` | four rare minor purposes (9–22 rows each) || `other_debtors` | `A102` + `A103` → `A102_103` | both mean "third-party backing exists" || `job` | `A171` + `A172` → `A171_172` | both are unskilled categories |After collapsing, the model converges in **7 iterations** and the largest fittedprobability is 0.9934 rather than 1.0000.

In [ ]:
COLLAPSE = {    "credit_history": {"A30":"A30_31", "A31":"A30_31"},    "purpose":        {"A44":"A4_other", "A45":"A4_other",                       "A48":"A4_other", "A410":"A4_other"},    "other_debtors":  {"A102":"A102_103", "A103":"A102_103"},    "job":            {"A171":"A171_172", "A172":"A171_172"},}# Evidence that collapsing is necessary. The split below uses exactly the same# parameters as the real split later (it depends only on the index and the labels,# neither of which collapsing changes), so these ARE the training counts the model sees._idx_tr, _ = train_test_split(df_all.index.values, test_size=TEST_SIZE,                              random_state=SEED, stratify=df_all["approval"].values)_tr = df_all.loc[np.sort(_idx_tr)]print("sparse levels, counted in the 700-row TRAINING split (seed %d):" % SEED)for col in COLLAPSE:    vc = _tr[col].value_counts().sort_index()    for lvl in vc.index:        n = int(vc[lvl])        if n < 30:            rate = _tr.loc[_tr[col] == lvl, "approval"].mean()            flag = "   <-- PERFECT SEPARATION: MLE diverges" if rate in (0.0, 1.0) else ""            print(f"  {col:16s} {lvl:8s} n={n:3d}  approval_rate={rate:.3f}{flag}")print("\n(for reference, the same levels counted on all 1000 rows:)")for col in COLLAPSE:    vc = raw[col].value_counts().sort_index()    rare = {lvl:int(vc[lvl]) for lvl in vc.index if vc[lvl] < 30}    if rare: print(f"  {col:16s} {rare}")def collapse_levels(df):    df = df.copy()    for col, mapping in COLLAPSE.items():        df[col] = df[col].replace(mapping)    return dfdf = add_derived(collapse_levels(raw))print("\nlevel counts after collapsing:")for col in COLLAPSE:    print(f"  {col:16s}", df[col].value_counts().sort_index().to_dict())

### 2.5 Feature scalingThe seven numeric attributes are standardised to mean 0 / SD 1. Two reasons:1. **Comparability.** Raw units differ by four orders of magnitude (`credit_amount`   in DM vs `installment_rate` on a 1–4 scale), so unscaled logistic coefficients cannot   be compared against each other. After scaling, every numeric coefficient reads as   "change in log-odds per one standard deviation".2. **Conditioning.** Newton–Raphson converges faster and more stably on a well-scaled   design matrix.**Scaling statistics are computed on the training set only** and then applied to thetest set, so no test information leaks into training. One-hot encoding is label-free andcan safely be fitted on the full frame, which also guarantees train and test shareidentical columns.

In [ ]:
# one-hot on the full frame (label-free -> no leakage, and guarantees identical columns)X_cat = pd.get_dummies(df[CATEGORICAL].astype(str), drop_first=True, dtype=float)X_num_raw = df[NUMERIC].astype(float)y_all = df["approval"].valuesidx_train, idx_test = train_test_split(df.index.values, test_size=TEST_SIZE,                                       random_state=SEED, stratify=y_all)idx_train, idx_test = np.sort(idx_train), np.sort(idx_test)# scaling parameters from TRAIN ONLYmu = X_num_raw.loc[idx_train].mean()sd = X_num_raw.loc[idx_train].std(ddof=0)X_num = (X_num_raw - mu) / sdX = pd.concat([X_num, X_cat], axis=1)X_train, X_test = X.loc[idx_train], X.loc[idx_test]y_train, y_test = df.loc[idx_train,"approval"].values, df.loc[idx_test,"approval"].valuesprint("train:", X_train.shape, " test:", X_test.shape, " features:", X.shape[1])print("approval rate  train=%.4f  test=%.4f" % (y_train.mean(), y_test.mean()))print("\ntraining-set scaling parameters:")print(pd.DataFrame({"mean": mu, "sd": sd}).round(3).to_string())

---## 3. Logistic regression by maximum likelihoodNewton–Raphson (IRLS) with a step-halving line search. Standard errors come from theinverse observed Fisher information $(X^\top W X)^{-1}$ with $W = \mathrm{diag}(p(1-p))$;z-statistics, p-values and 95% CIs follow from the normal approximation.

In [ ]:
class LogitMLE:    '''Unpenalised logistic regression by Newton-Raphson with analytic inference.'''    def __init__(self, ridge=0.0):        self.ridge = ridge    def fit(self, Xdf, y, max_iter=200, tol=1e-10):        self.columns = list(Xdf.columns)        Xd = np.hstack([np.ones((len(Xdf),1)), np.asarray(Xdf, dtype=float)])        y  = np.asarray(y, dtype=float)        n, k = Xd.shape        b = np.zeros(k)        def nll(bb):            e = Xd @ bb            ll = np.sum(y*e - np.logaddexp(0.0, e))          # stable log-likelihood            return -(ll - 0.5*self.ridge*np.sum(bb[1:]**2))        for it in range(max_iter):            eta = Xd @ b            p = 1.0/(1.0+np.exp(-eta))            W = np.clip(p*(1-p), 1e-12, None)            g = Xd.T @ (y - p) - self.ridge*np.r_[0, b[1:]]            H = Xd.T @ (Xd * W[:,None]) + self.ridge*np.diag(np.r_[0, np.ones(k-1)])            try:                step = np.linalg.solve(H, g)            except np.linalg.LinAlgError:                step = np.linalg.lstsq(H, g, rcond=None)[0]            f0 = nll(b); lam = 1.0; b_new = b + step            while lam > 1e-6 and (not np.isfinite(nll(b_new)) or nll(b_new) > f0):                lam *= 0.5; b_new = b + lam*step            b = b_new            if np.max(np.abs(lam*step)) < tol:                break        self.n_iter_ = it + 1        self.converged_ = np.max(np.abs(lam*step)) < 1e-6        self.beta = b        eta = Xd @ b; p = 1/(1+np.exp(-eta)); W = np.clip(p*(1-p), 1e-12, None)        H = Xd.T @ (Xd*W[:,None]) + self.ridge*np.diag(np.r_[0, np.ones(k-1)])        self.cov = np.linalg.pinv(H)        self.se  = np.sqrt(np.clip(np.diag(self.cov), 0, None))        self.z   = np.divide(self.beta, self.se, out=np.zeros_like(self.beta), where=self.se>0)        self.p   = 2*(1 - stats.norm.cdf(np.abs(self.z)))        self.ci_lo = self.beta - 1.959963985*self.se        self.ci_hi = self.beta + 1.959963985*self.se        self.loglik = float(np.sum(y*np.log(np.clip(p,1e-15,1)) +                                   (1-y)*np.log(np.clip(1-p,1e-15,1))))        self.max_abs_p_ = float(p.max())        return self    def _align(self, Xdf):        return np.asarray(Xdf.reindex(columns=self.columns, fill_value=0.0), dtype=float)    def predict_proba(self, Xdf):        Xd = np.hstack([np.ones((len(Xdf),1)), self._align(Xdf)])        return 1.0/(1.0+np.exp(-(Xd @ self.beta)))    def table(self):        return pd.DataFrame({"coef":self.beta, "std_err":self.se, "z":self.z,                             "p_value":self.p, "ci_2.5%":self.ci_lo, "ci_97.5%":self.ci_hi},                            index=["(intercept)"] + self.columns)def ols_fit(Xdf, y):    Xd = np.hstack([np.ones((len(Xdf),1)), np.asarray(Xdf, dtype=float)])    beta, *_ = np.linalg.lstsq(Xd, np.asarray(y, dtype=float), rcond=None)    return betadef ols_predict(beta, Xdf, cols):    Xd = np.hstack([np.ones((len(Xdf),1)),                    np.asarray(Xdf.reindex(columns=cols, fill_value=0.0), dtype=float)])    return Xd @ betaprint("estimator defined")

---# Part 1 — Build and compare the two models* **Model A (OLS):** linear probability model on `approval`, then classified at the  **median predicted value** on the test set.* **Model B (Logistic):** classified at the **fixed 0.5 cutoff**.For German Credit the positive class is *approval*, so the brief's "high risk" flag mapsto **"flagged positive / approve"** here; credit risk is its complement. The comparisonbelow is between the two *thresholding rules*, as the brief specifies.

In [ ]:
# ---------- Model A: OLS (linear probability model) ----------beta_A   = ols_fit(X_train, y_train)predA_tr = ols_predict(beta_A, X_train, list(X.columns))predA_te = ols_predict(beta_A, X_test,  list(X.columns))median_A = float(np.median(predA_te))classA   = (predA_te >= median_A).astype(int)r2 = 1 - ((y_train-predA_tr)**2).sum()/((y_train-y_train.mean())**2).sum()n_oob = int(((predA_te < 0) | (predA_te > 1)).sum())print("=== MODEL A (OLS) ===")print("train R^2                 : %.4f" % r2)print("test prediction range     : [%.4f, %.4f]" % (predA_te.min(), predA_te.max()))print("predictions outside [0,1] : %d of 300 (%.1f%%)" % (n_oob, n_oob/3))print("MEDIAN cutoff             : %.4f" % median_A)print("flagged approve           : %d (%.1f%%)" % (classA.sum(), 100*classA.mean()))print("accuracy vs truth         : %.4f" % accuracy_score(y_test, classA))

In [ ]:
# ---------- Model B: logistic regression ----------modelB = LogitMLE().fit(X_train, y_train)probB  = modelB.predict_proba(X_test)classB = (probB >= 0.5).astype(int)print("=== MODEL B (Logistic MLE) ===")print("converged                 : %s in %d iterations" % (modelB.converged_, modelB.n_iter_))print("max fitted probability    : %.4f  (no separation)" % modelB.max_abs_p_)print("log-likelihood            : %.4f" % modelB.loglik)print("test probability range    : [%.4f, %.4f]" % (probB.min(), probB.max()))print("median predicted prob     : %.4f" % np.median(probB))print("flagged approve at 0.50   : %d (%.1f%%)" % (classB.sum(), 100*classB.mean()))print("accuracy vs truth         : %.4f" % accuracy_score(y_test, classB))print("AUC                       : %.4f" % roc_auc_score(y_test, probB))

In [ ]:
# ---------- agreement between the two thresholding rules ----------cm = confusion_matrix(classA, classB, labels=[0,1])cm_df = pd.DataFrame(cm, index=["Model A: reject","Model A: approve"],                         columns=["Model B: reject","Model B: approve"])mismatch = int((classA != classB).sum())print("=== AGREEMENT MATRIX (rows = Model A @ median, cols = Model B @ 0.50) ===")print(cm_df.to_string())print()print("mismatched cases          : %d of %d = %.2f%%" % (mismatch, len(classA), 100*mismatch/len(classA)))print("A=approve but B=reject    : %d" % int(((classA==1)&(classB==0)).sum()))print("A=reject  but B=approve   : %d" % int(((classA==0)&(classB==1)).sum()))# the disagreement is one-directional because the two scores rank cases almost identicallyrho = stats.spearmanr(predA_te, probB).statisticprint("\nSpearman rank correlation between the two scores: %.4f" % rho)print("-> both rules rank applicants the same way; they differ only in WHERE they cut.")mm = classA != classBprint("\namong the %d mismatches: Model A correct %d times, Model B correct %d times"      % (mismatch, int((classA[mm]==y_test[mm]).sum()), int((classB[mm]==y_test[mm]).sum())))

In [ ]:
# ---------- Figure: the two thresholding rules ----------OK = {"blue":"#0072B2","orange":"#E69F00","green":"#009E73","verm":"#D55E00",      "purple":"#CC79A7","ink":"#1a1a1a","mute":"#6b6b6b","grid":"#d9d9d9"}plt.rcParams.update({"font.size":9,"axes.edgecolor":OK["mute"],"text.color":OK["ink"], "xtick.color":OK["mute"],"ytick.color":OK["mute"],"axes.spines.top":False, "axes.spines.right":False,"figure.dpi":140,"axes.grid":True,"grid.color":OK["grid"], "grid.linewidth":.6,"grid.alpha":.7,"savefig.bbox":"tight","savefig.facecolor":"white"})fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.9))ax = axes[0]ax.hist(predA_te, bins=26, color=OK["blue"], alpha=.8, edgecolor="white", linewidth=.7)ax.set_ylim(0, 27); ax.axvline(median_A, color=OK["verm"], lw=2.1)ax.axvline(0, color=OK["mute"], lw=1, ls=":"); ax.axvline(1, color=OK["mute"], lw=1, ls=":")ax.annotate(f"median = {median_A:.4f}\nflags exactly 50%", (median_A, 26.2), xytext=(-10,0),            textcoords="offset points", ha="right", va="top", fontsize=8, color=OK["verm"])ax.annotate(f"{n_oob} predictions ({n_oob/3:.1f}%)\nlie outside [0,1] — impossible\nas probabilities",            (1.18, 6.5), fontsize=7.6, color=OK["mute"], ha="center",            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=OK["grid"]))ax.set_xlabel("Model A (OLS) predicted value"); ax.set_ylabel("count"); ax.set_xlim(-.05,1.45)ax.set_title("Model A: linear regression output", fontsize=10.2, loc="left")ax = axes[1]ax.hist(probB, bins=26, color=OK["green"], alpha=.8, edgecolor="white", linewidth=.7)ax.set_ylim(0, 55); ax.axvline(0.5, color=OK["verm"], lw=2.1)ax.annotate(f"fixed cutoff 0.50\nflags {int(classB.sum())} = {100*classB.mean():.1f}%", (0.5, 53),            xytext=(9,0), textcoords="offset points", ha="left", va="top", fontsize=8, color=OK["verm"])ax.set_xlabel("Model B (logistic) predicted probability"); ax.set_xlim(0,1)ax.set_title("Model B: logistic regression output", fontsize=10.2, loc="left")fig.suptitle(f"Two thresholding rules on the same test set disagree on {mismatch} of 300 cases "             f"({100*mismatch/len(classA):.2f}%)", fontsize=10.8, x=.004, ha="left", y=1.03)plt.savefig("fig4_part1_thresholds.png"); plt.show()

---# Part 2.1 — Simpson's Paradox huntCandidate demographic split variables are searched systematically for a predictor whosecoefficient **reverses sign** between subgroups. A parsimonious 7-attributespecification is used here because subgroup models are fitted on as few as 310 rows, and42 parameters on 310 rows would produce confidence intervals too wide to interpret.

In [ ]:
CORE_NUM = ["duration","credit_amount","age","installment_rate"]CORE_CAT = ["checking_status","credit_history","savings_status"]# scaling for this part uses the full data (this is a descriptive/structural analysis,# not a predictive evaluation, so there is no train/test leakage concern)mu_c = df[CORE_NUM].astype(float).mean()sd_c = df[CORE_NUM].astype(float).std(ddof=0)AGE_SD = float(sd_c["age"])def build_core(d):    Xc = pd.get_dummies(d[CORE_CAT].astype(str), drop_first=True, dtype=float)    Xn = (d[CORE_NUM].astype(float) - mu_c) / sd_c    Xo = pd.concat([Xn, Xc], axis=1)    return Xo.loc[:, Xo.std(ddof=0) > 0]      # drop degenerate columns inside a subgroupdef fit_core(d):    return LogitMLE().fit(build_core(d), d["approval"].values)df["age_group"] = np.where(df["age"] <= 30, "young(<=30)", "older(>30)")df["housing_b"] = np.where(df["housing"] == "A152", "owner", "non_owner")df["job_b"]     = np.where(df["job"] == "A174", "mgmt_highqual", "other_job")SPLITS = {"sex":("female","male"), "age_group":("young(<=30)","older(>30)"),          "housing_b":("owner","non_owner"), "job_b":("mgmt_highqual","other_job"),          "foreign_worker":("A201","A202")}m_full = fit_core(df); t_full = m_full.table()print("=== scanning for sign reversals ===\n")for var,(la,lb) in SPLITS.items():    dA, dB = df[df[var]==la], df[df[var]==lb]    if min(len(dA), len(dB)) < 60:        print(f"[skipped] {var}: subgroup too small (nA={len(dA)}, nB={len(dB)})\n"); continue    mA, mB = fit_core(dA), fit_core(dB); tA, tB = mA.table(), mB.table()    print(f"--- {var}: A={la} (n={len(dA)}), B={lb} (n={len(dB)})  converged={mA.converged_}/{mB.converged_}")    found = False    for c in [c for c in t_full.index if c in tA.index and c in tB.index and c != "(intercept)"]:        if np.sign(tA.loc[c,"coef"]) != np.sign(tB.loc[c,"coef"]):            star = " ***" if min(tA.loc[c,"p_value"], tB.loc[c,"p_value"]) < 0.10 else ""            print(f"    REVERSAL  {c:22s} full={t_full.loc[c,'coef']:+.4f}  "                  f"A={tA.loc[c,'coef']:+.4f}(p={tA.loc[c,'p_value']:.3f})  "                  f"B={tB.loc[c,'coef']:+.4f}(p={tB.loc[c,'p_value']:.3f}){star}")            found = True    if not found: print("    (no sign reversal)")    print()

**Selected finding: `age`, split by `sex`.** It is the only reversal where both subgroupsare adequately sized, both models converge, and one side is strongly significant.

In [ ]:
m_F = fit_core(df[df.sex=="female"]); m_M = fit_core(df[df.sex=="male"])t_F, t_M = m_F.table(), m_M.table()print("=== AGE coefficient across the three models (per 1 SD = %.2f years) ===" % AGE_SD)hdr = f"{'model':22s}{'n':>6s}{'coef':>10s}{'SE':>9s}{'z':>8s}{'p':>9s}{'95% CI':>22s}{'OR/10yr':>10s}"print(hdr); print("-"*len(hdr))simpson = {}for name, tbl, n in [("Full (pooled)", t_full, len(df)),                     ("Subgroup A: female", t_F, int((df.sex=="female").sum())),                     ("Subgroup B: male",   t_M, int((df.sex=="male").sum()))]:    r = tbl.loc["age"]; or10 = np.exp(r["coef"]/AGE_SD*10)    print(f"{name:22s}{n:6d}{r['coef']:+10.4f}{r['std_err']:9.4f}{r['z']:8.3f}"          f"{r['p_value']:9.4f}   [{r['ci_2.5%']:+.4f},{r['ci_97.5%']:+.4f}]{or10:10.3f}")    simpson[name] = dict(n=n, **{k: float(r[k]) for k in                         ["coef","std_err","z","p_value","ci_2.5%","ci_97.5%"]}, OR_per_10yr=float(or10))

In [ ]:
# ---- formal test: is the difference real, or just subgroup noise? ----X_int = build_core(df).copy()X_int["is_female"]    = df["is_female"].valuesX_int["age_x_female"] = ((df["age"].astype(float)-mu_c["age"])/sd_c["age"]).values * df["is_female"].valuesm_int = LogitMLE().fit(X_int, df["approval"].values)t_int = m_int.table()m_red = LogitMLE().fit(X_int.drop(columns=["age_x_female"]), df["approval"].values)lr_stat = 2*(m_int.loglik - m_red.loglik)lr_p    = 1 - stats.chi2.cdf(lr_stat, 1)print("=== pooled model with sex and an age x sex interaction ===")print(t_int.loc[["age","is_female","age_x_female"]].to_string(float_format=lambda v: f"{v:.4f}"))ix = t_int.loc["age_x_female"]print(f"\ninteraction  age x female = {ix['coef']:+.4f}  SE={ix['std_err']:.4f}  "      f"z={ix['z']:.3f}  p={ix['p_value']:.4f}  95% CI [{ix['ci_2.5%']:+.4f}, {ix['ci_97.5%']:+.4f}]")print(f"likelihood-ratio test: chi2(1) = {lr_stat:.4f}, p = {lr_p:.4f}")print(f"\nimplied male age slope   = {t_int.loc['age','coef']:+.4f}")print(f"implied female age slope = {t_int.loc['age','coef']+ix['coef']:+.4f}")print("-> the subgroup difference is statistically significant, not sampling noise.")

In [ ]:
# ---- raw data check: the paradox is visible without any model ----df["age_band"] = pd.cut(df["age"], [18,25,30,40,50,100],                        labels=["19-25","26-30","31-40","41-50","51+"])raw_tab = df.pivot_table(index="age_band", columns="sex", values="approval",                         aggfunc=["mean","count"], observed=True)print("=== actual approval rate by age band and sex (no model involved) ===")print(raw_tab.to_string(float_format=lambda v: f"{v:.3f}"))

In [ ]:
# ---------- Figure: Simpson's paradox ----------fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.2))ax = axes[0]order = ["Full (pooled)","Subgroup A: female","Subgroup B: male"]cols  = [OK["mute"], OK["purple"], OK["blue"]]for i,(nm,col) in enumerate(zip(order, cols)):    r = simpson[nm]    ax.plot([r["ci_2.5%"], r["ci_97.5%"]], [i,i], color=col, lw=2.8, solid_capstyle="round")    ax.plot([r["coef"]], [i], "o", color=col, ms=9.5, mec="white", mew=1.6, zorder=3)    ax.annotate(f"{r['coef']:+.3f}   (p={r['p_value']:.3f})", (r["coef"], i),                textcoords="offset points", xytext=(0,15), ha="center", fontsize=8.4)ax.axvline(0, color=OK["verm"], lw=1.3, ls=(0,(4,3)))ax.set_yticks(range(3)); ax.set_yticklabels([f"{n}\n(n={simpson[n]['n']})" for n in order], fontsize=8.4)ax.set_xlabel(f"logistic coefficient on AGE   (per 1 SD = {AGE_SD:.2f} years)")ax.set_title("Age effect reverses sign between subgroups", fontsize=10.4, loc="left", pad=14)ax.set_xlim(-.45, 1.05); ax.set_ylim(2.75, -0.75); ax.grid(axis="y", visible=False)ax.text(.985,.055, f"pooled interaction  age × female = {ix['coef']:+.3f}\n"        f"p = {ix['p_value']:.4f}   ·   LR χ²(1) = {lr_stat:.2f}",        transform=ax.transAxes, ha="right", va="bottom", fontsize=8,        bbox=dict(boxstyle="round,pad=0.4", fc="#f4f4f2", ec=OK["grid"]))ax = axes[1]gF = df[df.sex=="female"].groupby("age_band", observed=True)["approval"].agg(["mean","count"])gM = df[df.sex=="male"  ].groupby("age_band", observed=True)["approval"].agg(["mean","count"])ax.plot(range(5), gF["mean"], "-o", color=OK["purple"], lw=2.3, ms=7, mec="white", mew=1.5, label="Female")ax.plot(range(5), gM["mean"], "-o", color=OK["blue"],   lw=2.3, ms=7, mec="white", mew=1.5, label="Male")for i in range(5):    fv, mv = gF["mean"].iloc[i], gM["mean"].iloc[i]    fo, mo = ((-17,11) if fv < mv else (12,-18))    ax.annotate(f"n={gF['count'].iloc[i]}", (i,fv), textcoords="offset points",                xytext=(0,fo), ha="center", fontsize=7, color=OK["purple"])    ax.annotate(f"n={gM['count'].iloc[i]}", (i,mv), textcoords="offset points",                xytext=(0,mo), ha="center", fontsize=7, color=OK["blue"])ax.set_xticks(range(5)); ax.set_xticklabels(["19-25","26-30","31-40","41-50","51+"])ax.set_xlabel("age band"); ax.set_ylabel("actual approval rate")ax.set_ylim(.46,.90); ax.set_xlim(-.45,4.45); ax.legend(frameon=False, fontsize=8.8, loc="lower right")ax.set_title("Raw approval rate: rises with age for women, flat for men", fontsize=10.4, loc="left", pad=14)plt.savefig("fig3_simpson.png"); plt.show()

---# Part 2.2 — Omitted-variable bias`credit_amount` is dropped. It is a textbook confounder here: it correlates **+0.625**with `duration` and **−0.271** with `installment_rate`, and it independently predictsapproval, so removing it forces those two variables to absorb its effect.

In [ ]:
print("correlation of credit_amount with the other numeric attributes:")print(df[NUMERIC].corr()["credit_amount"].drop("credit_amount").round(4).to_string())

In [ ]:
DROP = "credit_amount"keep = [c for c in X.columns if c != DROP]m_fullmod = LogitMLE().fit(X_train, y_train)m_reduced = LogitMLE().fit(X_train[keep], y_train)tF_, tR_  = m_fullmod.table(), m_reduced.table()lr2 = 2*(m_fullmod.loglik - m_reduced.loglik); lr2_p = 1 - stats.chi2.cdf(lr2, 1)pf = m_fullmod.predict_proba(X_test); pr = m_reduced.predict_proba(X_test[keep])auc_f, auc_r = roc_auc_score(y_test, pf), roc_auc_score(y_test, pr)acc_f = accuracy_score(y_test,(pf>=.5).astype(int)); acc_r = accuracy_score(y_test,(pr>=.5).astype(int))print("=== model fit: full vs reduced ===")print("log-likelihood   full=%.4f   reduced=%.4f" % (m_fullmod.loglik, m_reduced.loglik))print("LR test chi2(1) = %.4f, p = %.5f" % (lr2, lr2_p))print("test AUC         full=%.4f   reduced=%.4f   (change %+.4f)" % (auc_f, auc_r, auc_r-auc_f))print("test accuracy    full=%.4f   reduced=%.4f   (change %+.4f)" % (acc_f, acc_r, acc_r-acc_f))print("\n>>> the reduced model looks almost as good on every headline metric.")

In [ ]:
REPORT = ["duration","installment_rate","existing_credits","purpose_A43","purpose_A41","property_A124"]LABELS = {"duration":"Duration in months (per SD=12.06)",          "installment_rate":"Installment rate (per SD=1.12)",          "existing_credits":"No. existing credits (per SD=0.58)",          "purpose_A43":"Purpose: radio/television",          "purpose_A41":"Purpose: used car",          "property_A124":"Property: unknown/none"}rows=[]for c in REPORT:    b0, b1 = tF_.loc[c,"coef"], tR_.loc[c,"coef"]    rows.append(dict(predictor=LABELS[c], full=b0, reduced=b1, abs_change=b1-b0,                     pct_change=(b1-b0)/abs(b0)*100, shift_in_SE=(b1-b0)/tF_.loc[c,"std_err"],                     p_full=tF_.loc[c,"p_value"], p_reduced=tR_.loc[c,"p_value"]))ovb_table = pd.DataFrame(rows).set_index("predictor")print("=== before / after coefficients (all six are significant at p<0.10 in the full model) ===")print(ovb_table.to_string(float_format=lambda v: f"{v:+.4f}"))biggest = ovb_table["pct_change"].abs().idxmax()print(f"\nLARGEST SHIFT: {biggest}")print(f"   {ovb_table.loc[biggest,'full']:+.4f} -> {ovb_table.loc[biggest,'reduced']:+.4f}"      f"  = {ovb_table.loc[biggest,'pct_change']:+.1f}%  ({ovb_table.loc[biggest,'shift_in_SE']:.2f} SE)")

### Verifying the bias is not a coincidenceOmitted-variable-bias theory predicts both the **direction and approximate size** of eachshift:$$\text{bias}(\hat\beta_j) \;\approx\; \beta_{\text{omitted}} \times \delta_j$$where $\delta_j$ is the coefficient on predictor $j$ in an auxiliary regression of the*omitted* variable on all retained predictors. If the observed shifts match thisprediction, the movement is genuine confounding rather than numerical noise.

In [ ]:
beta_omitted = tF_.loc[DROP,"coef"]aux = pd.Series(ols_fit(X_train[keep], X_train[DROP].values)[1:], index=keep)chk=[]for c in REPORT:    predicted = beta_omitted * aux[c]    observed  = tR_.loc[c,"coef"] - tF_.loc[c,"coef"]    chk.append(dict(predictor=c,                    corr_with_amount=float(np.corrcoef(X_train[c], X_train[DROP])[0,1]),                    delta_j=aux[c], predicted_bias=predicted, observed_shift=observed,                    sign_matches=bool(np.sign(predicted)==np.sign(observed))))ovb_check = pd.DataFrame(chk).set_index("predictor")print("beta_omitted (credit_amount in the full logit) = %+.4f\n" % beta_omitted)print(ovb_check.to_string(float_format=lambda v: f"{v:+.4f}"))print("\nsign agreement: %d of %d predictors" % (ovb_check["sign_matches"].sum(), len(ovb_check)))

In [ ]:
# ---------- Figure: omitted-variable bias ----------from matplotlib.lines import Line2Dfig, ax = plt.subplots(figsize=(7.4, 3.9))ypos = np.arange(len(ovb_table))[::-1]for i,(nm,r) in zip(ypos, ovb_table.iterrows()):    ax.annotate("", xy=(r["reduced"],i), xytext=(r["full"],i),                arrowprops=dict(arrowstyle="-|>", color=OK["orange"], lw=1.9, shrinkA=0, shrinkB=0))    ax.plot([r["full"]],[i], "o", color=OK["blue"],   ms=7.5, mec="white", mew=1.4, zorder=3)    ax.plot([r["reduced"]],[i], "D", color=OK["orange"], ms=6.5, mec="white", mew=1.3, zorder=3)    ax.annotate(f"{r['pct_change']:+.1f}%", (max(r['full'],r['reduced']),i),                textcoords="offset points", xytext=(9,-3), fontsize=7.8)ax.axvline(0, color=OK["mute"], lw=1, ls=":")ax.set_yticks(ypos); ax.set_yticklabels(ovb_table.index, fontsize=8)ax.set_xlabel("logistic coefficient"); ax.set_xlim(-.95, 1.75); ax.grid(axis="y", visible=False)ax.legend(handles=[Line2D([],[],marker="o",ls="",color=OK["blue"],ms=7,label="Full model (all 20 attributes)"),                   Line2D([],[],marker="D",ls="",color=OK["orange"],ms=6,label="Reduced (credit_amount dropped)")],          frameon=False, fontsize=8, loc="lower right")ax.set_title("Dropping one confounder moves coefficients while test AUC falls only %.4f" % (auc_f-auc_r),             fontsize=10.2, loc="left", pad=8)plt.savefig("fig5_ovb.png"); plt.show()

---# Part 2.3 — Adversarial subset constructionTest cases where the model is **confident and wrong**: $P \ge 0.75$ but the applicantdefaulted, or $P \le 0.25$ but the applicant repaid.

In [ ]:
model_main = LogitMLE().fit(X_train, y_train)t_main = model_main.table()p_te   = model_main.predict_proba(X_test)test_df = df.loc[idx_test].copy(); test_df["p_hat"] = p_tehc_fp = (p_te >= 0.75) & (y_test == 0)   # confidently approved, actually bad credithc_fn = (p_te <= 0.25) & (y_test == 1)   # confidently rejected, actually good creditadv   = hc_fp | hc_fnprint("=== high-confidence errors on the %d-row test set ===" % len(test_df))print("P >= 0.75 but actually BAD  (confident false approval): %d" % hc_fp.sum())print("P <= 0.25 but actually GOOD (confident false rejection): %d" % hc_fn.sum())print("TOTAL                                                  : %d = %.2f%% of the test set"      % (adv.sum(), 100*adv.mean()))print()print("context: %d cases scored P>=0.75, of which %.1f%% were wrong"      % ((p_te>=.75).sum(), 100*hc_fp.sum()/max((p_te>=.75).sum(),1)))print("         %d cases scored P<=0.25, of which %.1f%% were wrong"      % ((p_te<=.25).sum(), 100*hc_fn.sum()/max((p_te<=.25).sum(),1)))adv_all = test_df[adv].copy()rng = np.random.default_rng(SEED)if len(adv_all) > 50:    ADV = adv_all.loc[np.sort(rng.choice(adv_all.index.values, 50, replace=False))]    print("\nmore than 50 exist -> randomly sampled 50 (rng seed %d)" % SEED)else:    ADV = adv_all    print("\nFEWER THAN 50 EXIST -> using all %d, as the brief instructs" % len(ADV))

In [ ]:
# ---------- demographics: adversarial subset vs full test set ----------def profile(d):    return {"n": len(d), "mean age": d.age.mean(), "median age": d.age.median(),            "% age <= 30": 100*(d.age<=30).mean(), "% female": 100*(d.sex=="female").mean(),            "mean duration (months)": d.duration.mean(), "mean credit amount (DM)": d.credit_amount.mean(),            "mean installment rate": d.installment_rate.mean(),            "% no checking acct (A14)": 100*(d.checking_status=="A14").mean(),            "% no/unknown savings (A65)": 100*(d.savings_status=="A65").mean(),            "% critical history (A34)": 100*(d.credit_history=="A34").mean(),            "% owns home": 100*(d.housing=="A152").mean(),            "% skilled/mgmt job": 100*(d.job.isin(["A173","A174"])).mean(),            "% foreign worker": 100*(d.foreign_worker=="A201").mean(),            "% actually approved": 100*d.approval.mean()}pa, pb = profile(ADV), profile(test_df)demo = pd.DataFrame({"adversarial subset": pa, "full test set": pb})demo["difference"] = demo["adversarial subset"] - demo["full test set"]print("=== DEMOGRAPHICS ===")print(demo.to_string(float_format=lambda v: f"{v:.2f}"))

In [ ]:
# ---------- shared characteristics ----------def traits(d):    return {"credit_amount <= 5000 DM":        (d.credit_amount<=5000),            "no third-party guarantor (A101)": (d.other_debtors=="A101"),            "duration <= 24 months":           (d.duration<=24),            "clean-looking history (A32/A34)": (d.credit_history.isin(["A32","A34"])),            "installment_rate >= 3":           (d.installment_rate>=3),            "owns home (A152)":                (d.housing=="A152"),            "age <= 35":                       (d.age<=35),            "no checking acct (A14) OR no savings (A65)":                ((d.checking_status=="A14")|(d.savings_status=="A65")),            "no checking account (A14)":       (d.checking_status=="A14"),            "critical history (A34)":          (d.credit_history=="A34")}ta, tb = traits(ADV), traits(test_df)tr = pd.DataFrame({"% of adversarial subset":{k:100*v.mean() for k,v in ta.items()},                   "% of full test set":     {k:100*v.mean() for k,v in tb.items()}})tr["lift (pp)"] = tr.iloc[:,0] - tr.iloc[:,1]print("=== SHARED CHARACTERISTICS ===")print(tr.sort_values("% of adversarial subset", ascending=False).to_string(float_format=lambda v: f"{v:.1f}"))FP = test_df[(p_te>=.75)&(y_test==0)]print("\n=== the %d confident FALSE APPROVALS (the costly errors: 5:1 cost matrix) ===" % len(FP))print("  %% with no checking account (A14)   : %.1f%%" % (100*(FP.checking_status=="A14").mean()))print("  %% with clean-looking history A32/A34: %.1f%%" % (100*FP.credit_history.isin(["A32","A34"]).mean()))print("  %% with A14 or A65 (absence of data) : %.1f%%"      % (100*((FP.checking_status=="A14")|(FP.savings_status=="A65")).mean()))print("  mean predicted P(approve)           : %.4f  (range %.3f - %.3f)"      % (FP.p_hat.mean(), FP.p_hat.min(), FP.p_hat.max()))

In [ ]:
# ---------- WHY: the coefficients that create the confidence ----------print("=== the largest positive coefficients in the model ===")top = t_main.drop("(intercept)").sort_values("coef", ascending=False).head(6)for c, r in top.iterrows():    print(f"  {c:30s} coef={r['coef']:+.4f}   odds ratio={np.exp(r['coef']):5.2f}   p={r['p_value']:.4f}")print("\n=== training-set approval rate by checking_status (what the model learned) ===")print(df.loc[idx_train].groupby("checking_status")["approval"].agg(["mean","count"]).round(3).to_string())print("\nThe two largest positive coefficients, A14 and A65, both mean ABSENCE OF A RECORD.")

In [ ]:
# ---------- Is this a fixable specification error? Test candidate interactions. ----------base_ll, base_err = model_main.loglik, int(adv.sum())cands = {"A14 x installment_rate":   X["checking_status_A14"]*X["installment_rate"],         "duration x credit_amount": X["duration"]*X["credit_amount"],         "A14 x credit_amount":      X["checking_status_A14"]*X["credit_amount"],         "A34 x duration":           X["credit_history_A34"]*X["duration"],         "A14 x duration":           X["checking_status_A14"]*X["duration"]}out=[]for nm, v in cands.items():    X2 = X.copy(); X2["interaction"] = v.values    m2 = LogitMLE().fit(X2.loc[idx_train], y_train)    p2 = m2.predict_proba(X2.loc[idx_test])    lr = 2*(m2.loglik - base_ll)    out.append(dict(interaction=nm, coef=m2.table().loc["interaction","coef"],                    p_value=m2.table().loc["interaction","p_value"],                    LR_chi2=lr, LR_p=1-stats.chi2.cdf(lr,1),                    test_AUC=roc_auc_score(y_test,p2),                    confident_errors=int((((p2>=.75)&(y_test==0))|((p2<=.25)&(y_test==1))).sum())))print("=== does adding a missing interaction fix the confident errors? ===")print(pd.DataFrame(out).sort_values("LR_chi2",ascending=False).to_string(index=False, float_format=lambda v:f"{v:+.4f}"))print(f"\nbaseline confident errors = {base_err}")print(">>> several interactions are statistically detectable, yet NONE reduces the confident")print(">>> error count. The failure is not a missing interaction term.")

In [ ]:
# ---------- The real mechanism: are the errors separable AT ALL? ----------band = p_te >= 0.75FPb, TPb = test_df[band & (y_test==0)], test_df[band & (y_test==1)]print("inside the high-confidence band (P>=0.75, n=%d): %d defaulted, %d did not"      % (band.sum(), len(FPb), len(TPb)))print("the model's own mean P:  defaulters=%.4f   non-defaulters=%.4f   (gap only %.4f)"      % (FPb.p_hat.mean(), TPb.p_hat.mean(), TPb.p_hat.mean()-FPb.p_hat.mean()))print("\n=== can ANY of the 20 recorded attributes separate them? ===")rows=[]for c in NUMERIC:    t_, p_ = stats.ttest_ind(FPb[c].astype(float), TPb[c].astype(float), equal_var=False)    rows.append((c,"numeric (Welch t)",FPb[c].mean(),TPb[c].mean(),p_))for c in CATEGORICAL:    ct = pd.crosstab(test_df[band][c], y_test[band])    try:   p_ = stats.chi2_contingency(ct)[1] if ct.shape[0] > 1 else np.nan    except Exception: p_ = np.nan    rows.append((c,"categorical (chi2)",np.nan,np.nan,p_))sep = pd.DataFrame(rows, columns=["attribute","test","mean_defaulters","mean_others","p_value"]).sort_values("p_value")print(sep.to_string(index=False, float_format=lambda v: f"{v:.4f}"))print("\nattributes significant at p<0.05 : %d of %d" % ((sep.p_value<0.05).sum(), len(sep)))print("significant after Bonferroni     : %d" % (sep.p_value < 0.05/len(sep)).sum())

In [ ]:
# ---------- honest check: out-of-fold, is there ANY signal inside the band? ----------Xb = np.asarray(X.loc[test_df.index[band]].reindex(columns=model_main.columns, fill_value=0.0), dtype=float)yb = y_test[band]aucs=[]for s in [224,1,2,3,4]:    oof = np.zeros(len(yb))    for tr_i, va_i in StratifiedKFold(5, shuffle=True, random_state=s).split(Xb, yb):        oof[va_i] = LogisticRegression(max_iter=5000, C=0.5).fit(Xb[tr_i], yb[tr_i]).predict_proba(Xb[va_i])[:,1]    aucs.append(roc_auc_score(yb, oof))in_sample = roc_auc_score(yb, LogitMLE(ridge=1.0).fit(pd.DataFrame(Xb, columns=model_main.columns), yb)                              .predict_proba(pd.DataFrame(Xb, columns=model_main.columns)))print("in-sample AUC refitting inside the band : %.4f   <-- overfitting (42 params / %d rows)" % (in_sample, len(yb)))print("honest 5-fold cross-validated AUC       : %.4f (sd %.4f) over 5 seeds" % (np.mean(aucs), np.std(aucs)))print("   per-seed:", ", ".join(f"{a:.4f}" for a in aucs))print("\n>>> chance is 0.50. The recorded features carry essentially NO signal for")print(">>> separating these cases. The model is confident because it has correctly learned")print(">>> a GROUP BASE RATE, not because it knows anything about these individuals.")

In [ ]:
# ---------- Figure: adversarial subset ----------fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.9))ax = axes[0]; bins = np.linspace(0,1,21)ax.hist(p_te[y_test==1], bins=bins, color=OK["green"], alpha=.72, edgecolor="white",        linewidth=.7, label="actually good credit (y=1)")ax.hist(p_te[y_test==0], bins=bins, color=OK["verm"],  alpha=.80, edgecolor="white",        linewidth=.7, label="actually bad credit (y=0)")ax.set_ylim(0,66)ax.axvline(.75, color=OK["ink"], lw=1.4, ls=(0,(4,3))); ax.axvline(.25, color=OK["ink"], lw=1.4, ls=(0,(4,3)))ax.annotate(f"p ≥ 0.75 but defaulted\n= {int(hc_fp.sum())} confident false approvals", (0.90,30),            ha="center", fontsize=7.6, color=OK["verm"],            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=OK["verm"], alpha=.95))ax.annotate(f"p ≤ 0.25 but repaid = {int(hc_fn.sum())}", (0.26,56), xytext=(6,0),            textcoords="offset points", ha="left", fontsize=7.6, color=OK["mute"])ax.set_xlabel("predicted P(approve)"); ax.set_ylabel("count")ax.legend(frameon=False, fontsize=7.8, loc="upper left")ax.set_title("Where the confident errors live", fontsize=10.2, loc="left")ax = axes[1]feats = ["credit_amount","duration","age","installment_rate"]labs  = ["credit amount\n(DM)","duration\n(months)","age\n(years)","installment\nrate"]pvals = [float(sep.loc[sep.attribute==f,"p_value"].iloc[0]) for f in feats]xs = np.arange(4); w = .36fv = [FPb[f].mean() for f in feats]; tv = [TPb[f].mean() for f in feats]nm_ = [max(a,b) for a,b in zip(fv,tv)]ax.bar(xs-w/2, [a/n for a,n in zip(fv,nm_)], w-.03, color=OK["verm"],       label=f"{len(FPb)} that defaulted", edgecolor="white", linewidth=1)ax.bar(xs+w/2, [a/n for a,n in zip(tv,nm_)], w-.03, color=OK["green"],       label=f"{len(TPb)} that did not", edgecolor="white", linewidth=1)for i in range(4):    ax.annotate(f"p = {pvals[i]:.2f}", (i,1.045), ha="center", fontsize=7.8, color=OK["mute"])    f_ = f"{fv[i]:,.0f}" if feats[i]=="credit_amount" else f"{fv[i]:.1f}"    t_ = f"{tv[i]:,.0f}" if feats[i]=="credit_amount" else f"{tv[i]:.1f}"    ax.annotate(f_, (i-w/2, fv[i]/nm_[i]-.06), ha="center", fontsize=6.8, color="white", va="top")    ax.annotate(t_, (i+w/2, tv[i]/nm_[i]-.06), ha="center", fontsize=6.8, color="white", va="top")ax.set_xticks(xs); ax.set_xticklabels(labs, fontsize=8); ax.set_ylim(0,1.20)ax.set_ylabel("group mean (larger group scaled to 1.0)")ax.legend(frameon=False, fontsize=7.8, loc="lower right",          title="inside the p ≥ 0.75 band", title_fontsize=7.6)ax.set_title("None of the 20 recorded features separates them", fontsize=10.2, loc="left")ax.grid(axis="x", visible=False)plt.savefig("fig6_adversarial.png"); plt.show()ADV.to_csv("adversarial_subset.csv")print("adversarial subset written to adversarial_subset.csv (%d rows)" % len(ADV))

---# Part 2.4 — Fairness trade-off analysisSensitive variable: **sex**. Group A = female, Group B = male. The decision threshold isswept over $t \in \{0.05, 0.15, \ldots, 0.95\}$ and three metrics are computed at eachvalue.| Metric | Definition | Ideal ||---|---|---|| Demographic Parity (DP) | $P(\hat Y{=}1 \mid A) / P(\hat Y{=}1 \mid B)$ | 1.0 || Equalized Odds (EO) | $\min(\mathrm{TPR}_A/\mathrm{TPR}_B,\ \mathrm{FPR}_A/\mathrm{FPR}_B)$ | 1.0 || Predictive Parity (PP) | $P(Y{=}1 \mid \hat Y{=}1, A) / P(Y{=}1 \mid \hat Y{=}1, B)$ | 1.0 |

In [ ]:
gA = (test_df["sex"] == "female").values    # Group AgB = ~gA                                    # Group Bprint("Group A (female): n=%d   actual approval rate=%.4f   mean predicted P=%.4f"      % (gA.sum(), y_test[gA].mean(), p_te[gA].mean()))print("Group B (male)  : n=%d   actual approval rate=%.4f   mean predicted P=%.4f"      % (gB.sum(), y_test[gB].mean(), p_te[gB].mean()))def group_rates(yhat, ytrue):    P, N = (ytrue==1), (ytrue==0)    return (yhat.mean(),            yhat[P].mean() if P.sum() else np.nan,            yhat[N].mean() if N.sum() else np.nan,            ytrue[yhat==1].mean() if (yhat==1).sum() else np.nan)THRESHOLDS = [0.05,0.15,0.25,0.35,0.45,0.55,0.65,0.75,0.85,0.95]rows=[]for t in THRESHOLDS:    yh = (p_te >= t).astype(int)    selA,tprA,fprA,ppvA = group_rates(yh[gA], y_test[gA])    selB,tprB,fprB,ppvB = group_rates(yh[gB], y_test[gB])    DP = selA/selB if selB>0 else np.nan    eo_tpr = tprA/tprB if tprB>0 else np.nan    eo_fpr = fprA/fprB if fprB>0 else np.nan    EO = np.nanmin([eo_tpr, eo_fpr])    PP = ppvA/ppvB if (ppvB and ppvB>0) else np.nan    rows.append(dict(t=t, sel_A=selA, sel_B=selB, DP=DP, TPR_A=tprA, TPR_B=tprB,                     FPR_A=fprA, FPR_B=fprB, EO=EO, PPV_A=ppvA, PPV_B=ppvB, PP=PP,                     accuracy=(yh==y_test).mean(), n_approved=int(yh.sum())))sweep = pd.DataFrame(rows)print("\n=== FULL SWEEP ===")print(sweep.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

In [ ]:
print("=== REPORT TABLE: the three fairness metrics ===")print(sweep[["t","DP","EO","PP","accuracy","n_approved"]].to_string(index=False,      float_format=lambda v: f"{v:.4f}"))sweep["max_deviation"] = sweep[["DP","EO","PP"]].sub(1).abs().max(axis=1)print("\n=== no single threshold satisfies all three ===")print(sweep[["t","DP","EO","PP","max_deviation"]].to_string(index=False, float_format=lambda v: f"{v:.4f}"))for m in ["DP","EO","PP"]:    i = (sweep[m]-1).abs().idxmin()    print(f"closest to parity on {m}: t={sweep.loc[i,'t']:.2f} ({sweep.loc[i,m]:.4f})")best = sweep.loc[sweep.max_deviation.idxmin()]print(f"\nbest compromise (minimises the worst deviation): t={best.t:.2f}, max deviation={best.max_deviation:.4f}")print(f"  but note it approves {int(best.n_approved)} of 300 applicants ({100*best.n_approved/300:.1f}%)")

In [ ]:
# ---------- quantified trade-off between consecutive thresholds ----------print("=== trade-off cost between consecutive thresholds ===")print(f"{'move':16s}{'ΔDP %':>10s}{'ΔEO %':>10s}{'cost |ΔEO/ΔDP|':>18s}")tro=[]for i in range(len(sweep)-1):    a, b = sweep.iloc[i], sweep.iloc[i+1]    if np.isnan(a.DP) or np.isnan(b.DP) or np.isnan(a.EO) or np.isnan(b.EO) or a.EO==0: continue    dDP = (b.DP-a.DP)/a.DP*100; dEO = (b.EO-a.EO)/a.EO*100    cost = abs(dEO/dDP) if dDP != 0 else np.nan    tro.append(dict(frm=a.t, to=b.t, dDP_pct=dDP, dEO_pct=dEO, cost=cost))    print(f"t={a.t:.2f} -> {b.t:.2f}  {dDP:+10.2f}{dEO:+10.2f}{cost:18.3f}")tradeoff = pd.DataFrame(tro)print("\n=== the headline trade-off, in the brief's requested form ===")a = sweep[sweep.t==0.25].iloc[0]; b = sweep[sweep.t==0.35].iloc[0]gain = (a.DP-b.DP)/b.DP*100; loss = (b.EO-a.EO)/b.EO*100print(f"Moving t from 0.35 to 0.25 improves DP from {b.DP:.4f} to {a.DP:.4f} (a {gain:.2f}% gain),")print(f"while EO falls from {b.EO:.4f} to {a.EO:.4f} (a {loss:.2f}% loss).")print(f"The trade-off cost is {loss:.2f}/{gain:.2f} = {loss/gain:.3f}.")

In [ ]:
# ---------- REQUIRED FIGURE: DP vs EO trade-off chart ----------OFFSETS = {0.05:(20,14), 0.15:(34,2), 0.25:(30,-10), 0.35:(-8,-20), 0.45:(-34,10),           0.55:(26,-6), 0.65:(-30,6), 0.75:(24,-4), 0.85:(-30,-2), 0.95:(28,8)}fig, ax = plt.subplots(figsize=(6.4, 5.8))ax.axhline(1.0, color=OK["grid"], lw=.9); ax.axvline(1.0, color=OK["grid"], lw=.9)ax.plot(sweep.DP, sweep.EO, "-", color=OK["mute"], lw=1.2, alpha=.5)sc = ax.scatter(sweep.DP, sweep.EO, s=80, c=sweep.t, cmap="viridis",                edgecolor="white", linewidth=1.7, zorder=4)for _, r in sweep.iterrows():    dx, dy = OFFSETS[round(r.t,2)]    ax.annotate(f"t={r.t:.2f}", (r.DP, r.EO), textcoords="offset points", xytext=(dx,dy),                ha="center", fontsize=8, zorder=6,                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=.82))ax.scatter([1.0],[1.0], marker="*", s=300, color=OK["verm"], edgecolor="white", lw=1.2, zorder=5)ax.annotate("ideal (1.0, 1.0)\n— reached only at t=0.05,\nwhere every applicant is approved",            xy=(1.0,1.0), xytext=(0.30,1.30), fontsize=8, color=OK["verm"], ha="left", va="center",            arrowprops=dict(arrowstyle="->", color=OK["verm"], lw=1.1, connectionstyle="arc3,rad=-0.22"))ax.annotate("as t rises both metrics\ndeteriorate together", xy=(0.72,0.34), xytext=(0.17,0.52),            fontsize=8, color=OK["mute"], ha="left",            arrowprops=dict(arrowstyle="->", color=OK["mute"], lw=1, connectionstyle="arc3,rad=0.2"))ax.set_xlim(0,1.5); ax.set_ylim(0,1.5); ax.set_aspect("equal")ax.set_xticks(np.arange(0,1.6,.25)); ax.set_yticks(np.arange(0,1.6,.25))ax.set_xlabel("Demographic Parity   DP = P(Ŷ=1 | female) / P(Ŷ=1 | male)")ax.set_ylabel("Equalized Odds   EO = min(TPR$_F$/TPR$_M$,  FPR$_F$/FPR$_M$)")ax.set_title("Fairness trade-off across decision thresholds\n"             "German Credit · sex as sensitive attribute · n=300 test set",             fontsize=10.5, loc="left", pad=10)cb = fig.colorbar(sc, ax=ax, shrink=.72, pad=.03); cb.set_label("decision threshold t", fontsize=8.5)cb.outline.set_visible(False); cb.set_ticks(sweep.t.values)plt.savefig("fig1_tradeoff_DP_EO.png"); plt.show()

In [ ]:
# ---------- supporting figure: all three metrics against t ----------fig, ax = plt.subplots(figsize=(7.0, 4.1))ax.axhline(1.0, color=OK["ink"], lw=.9, ls=(0,(4,3)), alpha=.55)for k, col, lab in [("DP",OK["blue"],"Demographic Parity"), ("EO",OK["orange"],"Equalized Odds"),                    ("PP",OK["green"],"Predictive Parity")]:    ax.plot(sweep.t, sweep[k], "-o", color=col, lw=2.1, ms=5.8, mec="white", mew=1.3, label=lab)ax.plot(sweep.t, sweep.accuracy, "-s", color=OK["mute"], lw=1.4, ms=4.6, mec="white",        mew=1.1, label="Overall accuracy", alpha=.85)ax.annotate("parity = 1.0", (0.055,1.0), xytext=(2,8), textcoords="offset points",            fontsize=7.8, color=OK["mute"], ha="left")ax.annotate("EO collapses to 0\nat t = 0.95", (0.95,0.0), xytext=(-10,30), textcoords="offset points",            fontsize=8, color=OK["orange"], ha="right", arrowprops=dict(arrowstyle="->", color=OK["orange"], lw=1))ax.annotate("PP crosses 1.0 and keeps rising:\nit gets 'fairer' exactly where DP and EO collapse",            (0.85,1.065), xytext=(-14,26), textcoords="offset points", fontsize=7.8,            color=OK["green"], ha="right", arrowprops=dict(arrowstyle="->", color=OK["green"], lw=1))ax.set_xlabel("decision threshold t"); ax.set_ylabel("ratio (female / male)")ax.set_xticks(sweep.t.values); ax.set_ylim(-.05,1.34)ax.set_title("The three fairness metrics move in different directions", fontsize=10.6, loc="left", pad=8)ax.legend(frameon=False, fontsize=8.2, ncol=2, loc="lower left")plt.savefig("fig2_metrics_vs_threshold.png"); plt.show()sweep.to_csv("fairness_sweep.csv", index=False)tradeoff.to_csv("fairness_tradeoff.csv", index=False)print("tables written: fairness_sweep.csv, fairness_tradeoff.csv")

---# Summary of every headline numberPrinted in one place so each figure quoted in the PDF report can be traced back to thecell that produced it.

In [ ]:
S = { "SUBMISSION": {"name":NAME, "enrolment":ENROLMENT, "course":COURSE}, "SETUP": {"dataset":"German Credit (UCI statlog)", "rows":len(df), "random_state":SEED,           "n_train":len(idx_train), "n_test":len(idx_test), "n_features":X.shape[1]}, "PART 1": {"OLS train R2":round(r2,4), "OLS median cutoff":round(median_A,4),            "OLS preds outside [0,1]":f"{n_oob} ({n_oob/3:.1f}%)",            "OLS accuracy":round(accuracy_score(y_test,classA),4),            "Logit accuracy":round(accuracy_score(y_test,classB),4),            "Logit AUC":round(roc_auc_score(y_test,probB),4),            "Logit median prob":round(float(np.median(probB)),4),            "A flags approve":int(classA.sum()), "B flags approve":int(classB.sum()),            "mismatches":f"{mismatch} ({100*mismatch/len(classA):.2f}%)",            "A=approve,B=reject":int(((classA==1)&(classB==0)).sum()),            "A=reject,B=approve":int(((classA==0)&(classB==1)).sum()),            "Spearman rank corr":round(float(rho),4)}, "PART 2.1": {"split":"sex", "predictor":"age",              "full coef":round(simpson["Full (pooled)"]["coef"],4),              "full p":round(simpson["Full (pooled)"]["p_value"],4),              "female coef":round(simpson["Subgroup A: female"]["coef"],4),              "female p":round(simpson["Subgroup A: female"]["p_value"],4),              "male coef":round(simpson["Subgroup B: male"]["coef"],4),              "male p":round(simpson["Subgroup B: male"]["p_value"],4),              "interaction coef":round(float(ix["coef"]),4),              "interaction p":round(float(ix["p_value"]),4),              "LR chi2(1)":round(float(lr_stat),4), "LR p":round(float(lr_p),4)}, "PART 2.2": {"dropped":DROP, "LR chi2(1)":round(float(lr2),4), "LR p":round(float(lr2_p),5),              "AUC full":round(auc_f,4), "AUC reduced":round(auc_r,4),              "AUC loss":round(auc_f-auc_r,4),              "largest shift":biggest,              "largest shift pct":round(float(ovb_table.loc[biggest,"pct_change"]),2),              "largest shift SE":round(float(ovb_table.loc[biggest,"shift_in_SE"]),3),              "OVB sign agreement":f"{int(ovb_check['sign_matches'].sum())}/{len(ovb_check)}"}, "PART 2.3": {"confident FP (p>=.75, y=0)":int(hc_fp.sum()),              "confident FN (p<=.25, y=1)":int(hc_fn.sum()),              "total":f"{int(adv.sum())} ({100*adv.mean():.2f}%)",              "subset used":len(ADV),              "FP mean p_hat":round(float(FP.p_hat.mean()),4),              "features separating errors":f"0 of {len(sep)}",              "smallest p":round(float(sep.p_value.min()),4),              "CV AUC inside band":f"{np.mean(aucs):.4f} (sd {np.std(aucs):.4f})"}, "PART 2.4": {"Group A":f"female n={int(gA.sum())}", "Group B":f"male n={int(gB.sum())}",              "DP at t=0.45":round(float(sweep[sweep.t==0.45].DP.iloc[0]),4),              "EO at t=0.45":round(float(sweep[sweep.t==0.45].EO.iloc[0]),4),              "PP at t=0.45":round(float(sweep[sweep.t==0.45].PP.iloc[0]),4),              "accuracy at t=0.45":round(float(sweep[sweep.t==0.45].accuracy.iloc[0]),4),              "DP at t=0.55":round(float(sweep[sweep.t==0.55].DP.iloc[0]),4),              "EO at t=0.55":round(float(sweep[sweep.t==0.55].EO.iloc[0]),4),              "accuracy at t=0.55":round(float(sweep[sweep.t==0.55].accuracy.iloc[0]),4),              "DP range":f"{sweep.DP.min():.4f} - {sweep.DP.max():.4f}",              "EO range":f"{sweep.EO.min():.4f} - {sweep.EO.max():.4f}",              "PP range":f"{sweep.PP.min():.4f} - {sweep.PP.max():.4f}"},}for section, vals in S.items():    print("="*78); print(section); print("="*78)    for k,v in vals.items(): print(f"  {k:34s} {v}")    print()